# Model Parameters Optimization

## Getting Data + Target encoding

In [9]:
import os
import pandas as pd

data_path = os.path.join('Data', 'trainVal.csv')
data = pd.read_csv(data_path, index_col=False)
X_trainval, y_trainval = data.drop('Air_Quality', axis=1), data['Air_Quality']

# map = {'Hazardous': 0, 'Poor': 1, 'Moderate': 2, 'Good': 3} # Ordinal Encoding (as data is Ordinal) / other approach
# y_trainval = y_trainval.map(map)

y_trainval = pd.get_dummies(y_trainval) # one hot encoding - no bias, we don't assume anything about data

y_trainval

,Good,Hazardous,Moderate,Poor
0,True,False,False,False
1,True,False,False,False
2,True,False,False,False
3,False,False,False,True
4,False,True,False,False
...,...,...,...,...
3995,False,False,True,False
3996,False,False,True,False
3997,False,False,False,True
3998,False,True,False,False


In [10]:
from utils.training import get_dataSet

dataset = get_dataSet(X_trainval, y_trainval)

type(dataset)

torch.utils.data.dataset.TensorDataset

# Optuna Study

In [12]:
import os
import shutil
from torch.utils.tensorboard import SummaryWriter
from utils.dir_managment import clean_dir

writer_path = os.path.join('runs', 'optuna')
clean_dir(writer_path)

os.makedirs(writer_path, exist_ok=True)
writer = SummaryWriter(writer_path)

Cleaning existing files at runs/optuna...


## Objective
**Prompt**:
```
For 4000 instance trianval dataset, suggest best ranges for this parameters.
The network is optimized with 5 fold cross validation by mean validation loss score.

Params to optimize ...
```

| Parameter | Suggested Range | Reasoning |
| :--- | :--- | :--- |
| **`lr`** | `1e-4` to `1e-2` | `5e-1` (0.5) is extremely high and will likely cause the loss to explode. |
| **`batch_size`** | `16` to `128` | With 4,000 samples, a batch of 1 is too noisy; 256 is nearly 10% of your training fold. |
| **`beta1`** | `0.85` to `0.95` | This controls momentum. The default is 0.9. |
| **`beta2`** | `0.99` to `0.9999` | This controls the moving average of squared gradients. |

The goal is to find best parameters for **100** Epochs

In [4]:
# Paste this to console to see Tenosrboard
# tensorboard --logdir Lab6_7-project/runs/optuna

In [5]:
import numpy as np
import optuna
from functools import partial

from torch import nn
from torch import optim
from torch.utils.data import TensorDataset
from sklearn.model_selection import KFold

from utils.training import *
from utils.MLPClassifier import MLPClassifier

N_EPOCHS = 100
optuna.logging.set_verbosity(optuna.logging.WARNING) # No Reason to se results as we have tensorboard
def objective_nn(trial:optuna.Trial, dataset:TensorDataset, input_dim:int, output_dim:int, writer:SummaryWriter):
    global N_EPOCHS

    # Network Params
    # Using step=16 or 32 helps keep dimensions hardware-friendly
    hidden_dim = trial.suggest_int('hidden_dim', 32, 512, step=32)
    n_hidden = trial.suggest_int('n_hidden', 1, 5)

    # Optimizer Params
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])

    # If using Adam, betas are usually stable.
    # Only tune them if you have a specific reason to deviate from defaults.
    beta1 = trial.suggest_float('beta1', 0.8, 0.99)
    beta2 = trial.suggest_float('beta2', 0.99, 0.9999)

    # Scheduler Params
    factor = trial.suggest_float('factor', 0.1, 0.5)
    patience = trial.suggest_int('patience', 3, 7)


    fold_loss = []
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_trainval)):

        # Loaders
        train_loader, val_loader = get_train_loaders(dataset, train_idx=train_idx, val_idx=val_idx, batch_size=batch_size)

        # Model, Optimizer, Criterion
        model = MLPClassifier(input_dim=input_dim, output_dim=output_dim, hidden_dim=hidden_dim, n_hidden=n_hidden)
        criterion = nn.CrossEntropyLoss() # nn.MSELoss (better if we choose ordinal encoding)
        optimizer = optim.Adam(lr=lr, params=model.parameters(), betas=(beta1, beta2))
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=factor, patience=patience)

        # Fold Training
        loss = train_one_fold(fold, model, train_loader, val_loader, optimizer, scheduler, criterion, n_epochs=N_EPOCHS, write_model_dir=None, writer=None)
        fold_loss.append(loss)

    avg_loss = np.mean(fold_loss)
    # Writing Params
    writer.add_hparams(
        hparam_dict={
        'trial': trial.number,
        'hidden_dim': hidden_dim,
        'n_hidden': n_hidden,
        'lr': lr,
        'batch_size': batch_size,
        'beta1': beta1,
        'beta2': beta2,
        'factor': factor,
        'patience': patience
    },
    metric_dict={'hparam/mean_loss': avg_loss})
    writer.flush()


    return avg_loss

objective = partial(objective_nn, dataset=dataset, input_dim=X_trainval.shape[1], output_dim=y_trainval.shape[1], writer=writer)
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=25, show_progress_bar=True)

  0%|          | 0/25 [00:00<?, ?it/s]

## Results

In [6]:
import pandas as pd
trials = study.trials_dataframe().sort_values(by=['value'], ascending=True)

trials.head(5)

,number,value,datetime_start,datetime_complete,duration,params_batch_size,params_beta1,params_beta2,params_factor,params_hidden_dim,params_lr,params_n_hidden,params_patience,state
17,17,0.389345,2026-04-24 17:52:57.768736,2026-04-24 18:01:12.760381,0 days 00:08:14.991645,16,0.908764,0.990107,0.426219,64,0.000583,1,6,COMPLETE
11,11,0.392655,2026-04-24 17:09:23.326986,2026-04-24 17:18:19.157520,0 days 00:08:55.830534,16,0.896374,0.991008,0.456822,224,0.000394,1,7,COMPLETE
12,12,0.393199,2026-04-24 17:18:19.158738,2026-04-24 17:27:07.741035,0 days 00:08:48.582297,16,0.879574,0.990197,0.484798,224,0.000623,1,6,COMPLETE
21,21,0.396431,2026-04-24 18:26:13.726784,2026-04-24 18:30:45.046166,0 days 00:04:31.319382,16,0.885955,0.990178,0.459596,224,0.000629,1,6,COMPLETE
8,8,0.396653,2026-04-24 16:51:16.113616,2026-04-24 16:59:50.935245,0 days 00:08:34.821629,16,0.834048,0.992670,0.173430,128,0.000450,2,6,COMPLETE


In [7]:
from datetime import datetime

now = datetime.now()
formatted_date = now.strftime("%d_%m_%Y(%H:%M)")

path = os.path.join('optuna_results', f'trial_{formatted_date}.csv')
os.makedirs('optuna_results', exist_ok=True)
trials.to_csv(path, index=False)